# 00 · Data cleaning & Standard Schema v4
Labels without manual annotation: TTRS gloss · YouTube **title/filename** · **speech in the clip** (YouTube th-orig captions with word timestamps, Thai Whisper where missing). Hard-to-use sources are excluded with a recorded reason.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'modules').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Tahoma', 'Leelawadee UI', 'DejaVu Sans']
from IPython.display import display, Image, Audio, Markdown
from modules.utils import ART, read_json
REP, V4, MAN4 = ART / 'reports', ART / 'v4', ART / 'manifests_v4'
pd.set_option('display.max_colwidth', 80)

In [2]:
clips = pd.read_parquet(MAN4 / 'clips_v4.parquet')
clips.groupby(['source','subset']).agg(clips=('clip_id','size'), usable=('usable_for', lambda s: (s!='').sum()))

clips  usable
source  subset                           
ttrs    ttrs                 5043    5036
youtube bigsign                44       0
        youtube_continuous      1       0
        youtube_sentence       57      38
        youtube_word          186      86

In [3]:
clips.exclude_reason.value_counts(dropna=False).to_frame('clips')

,clips
exclude_reason,
None,5160
"no usable label (no Thai speech captions/ASR hits, title is a topic)",77
"broadcast/song interpretation: caption = announcer speech with lag, 2-person layouts",45
"fingerspelling/alphabet drill (letters, not lexical signs)",40
ttrs_long_clip(>15s: explanation / several signs),6
song interpretation (rhythm-driven signing),2
test_entry,1


In [4]:
clips[clips.label_source=='title'][['title','label','duration_s']].head(30)

,title,label,duration_s
5089,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/ดุด่า,ดุด่า,4.133
5090,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/ช่วย,ช่วย,5.062
5092,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/นอนไม่หลับ,นอนไม่หลับ,7.105
5096,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/ปัสสาวะ,ปัสสาวะ,4.133
5097,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/ขับรถยนต์,ขับรถยนต์,5.132
5099,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/นอนหลับ,นอนหลับ,7.105
5110,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/บอกลา,บอกลา,5.132
5111,ภาษามือไทยในชีวิตประจำวัน/หมวดร่างกาย/หูหนวก,หูหนวก,4.133
5117,ภาษามือไทย-หมวดทักทาย-ขอบคุณ 1,ขอบคุณ,2.647
5118,ภาษามือไทยในชีวิตประจำวัน/หมวดกิริยาอาการ/ถอดเสื้อ,ถอดเสื้อ,5.132


In [5]:
sw = pd.read_parquet(MAN4 / 'speech_words.parquet')
print('speech hits:', len(sw), 'distinct lexicon words:', sw.word.nunique(), '| sources:', sw.source.value_counts().to_dict())
sw.word.value_counts().head(25).to_frame('mentions')

speech hits: 7468 distinct lexicon words: 819 | sources: {'vtt_th-orig': 7248, 'vtt_th': 220}


,mentions
word,
เรา,402
คำ,289
ใน,266
เลย,229
ภาษามือ,220
และ,174
แบบ,141
มือ,133
หรือ,132


In [6]:
words = ['ฉัน','ปลอบใจ','เพื่อน','ร้องไห้','ไป','กิน','ข้าว','ด้วยกัน','ไหม']
ttrs = clips[clips.source=='ttrs']
pd.DataFrame([dict(word=w, ttrs_clips=int((ttrs.label==w).sum()), youtube_speech_mentions=int((sw.word==w).sum()), youtube_videos=int(sw[sw.word==w].clip_id.nunique())) for w in words])

,word,ttrs_clips,youtube_speech_mentions,youtube_videos
0,ฉัน,2,41,14
1,ปลอบใจ,1,0,0
2,เพื่อน,1,30,14
3,ร้องไห้,2,0,0
4,ไป,0,0,0
5,กิน,1,23,10
6,ข้าว,2,12,7
7,ด้วยกัน,1,11,9
8,ไหม,0,0,0
